## Intermediate Level
### Task 1

# Analysis of the Air Quality Index (AQI) in Delhi

### Step 1: Define Research Questions
We'll answer these:

What are the most dominant pollutants in Delhi?

How does AQI vary across months/seasons?

Which pollutants contribute most to AQI spikes?

Are there trends showing improvement or worsening over time?

### Step 2: Load and Inspect the Dataset

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

In [ ]:
import pandas as pd
df = pd.read_csv("C:\\Users\\INTEL\\Downloads\\delhiaqi.csv")
print(df.columns)
df.head()


### Step 3: Data Cleaning & Preprocessing

In [ ]:
# Check basic info
df.info()

# Check for null values
df.isnull().sum()

# Convert 'date' column to datetime
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Drop rows where date is missing
df.dropna(subset=['date'], inplace=True)

# Extract time-based features
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year

# Define season from month
def season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Summer'
    elif month in [6, 7, 8]:
        return 'Monsoon'
    else:
        return 'Post-Monsoon'

df['season'] = df['month'].apply(season)


### Step 4: Exploratory Data Analysis (EDA)

#### a. PM2.5 Over Time

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 5))
sns.lineplot(x='date', y='pm2_5', data=df)
plt.title('PM2.5 Trend in Delhi Over Time')
plt.xlabel('Date')
plt.ylabel('PM2.5 (µg/m³)')
plt.show()


#### b.  Seasonal Variation of PM2.5

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='season', y='pm2_5', data=df, palette='Set2')
plt.title('Seasonal Variation of PM2.5')
plt.xlabel('Season')
plt.ylabel('PM2.5 (µg/m³)')
plt.show()


#### c. Monthly Average PM2.5

In [ ]:
monthly_avg = df.groupby('month')['pm2_5'].mean()

plt.figure(figsize=(10, 5))
monthly_avg.plot(kind='bar', color='skyblue')
plt.title('Average Monthly PM2.5 Levels')
plt.xlabel('Month')
plt.ylabel('PM2.5 (µg/m³)')
plt.xticks(rotation=0)
plt.show()


####  d. PM10 Over Time

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 5))
sns.lineplot(x='date', y='pm10', data=df)
plt.title('PM10 Trend in Delhi Over Time')
plt.xlabel('Date')
plt.ylabel('PM10 (µg/m³)')
plt.show()


#### e. Seasonal Variation of PM10

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='season', y='pm10', data=df, palette='Set2')
plt.title('Seasonal Variation of PM10')
plt.xlabel('Season')
plt.ylabel('PM10 (µg/m³)')
plt.show()


#### f. Monthly Average PM10

In [ ]:
monthly_avg_pm10 = df.groupby('month')['pm10'].mean()

plt.figure(figsize=(10, 5))
monthly_avg_pm10.plot(kind='bar', color='coral')
plt.title('Average Monthly PM10 Levels')
plt.xlabel('Month')
plt.ylabel('PM10 (µg/m³)')
plt.xticks(rotation=0)
plt.show()


###  Step 5: Pollutant Contribution

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define correct column names
pollutants = ['pm2_5', 'pm10', 'no2', 'so2', 'co', 'o3']

# Summary statistics
print(df[pollutants].describe())

# Correlation matrix between pollutants
corr = df[pollutants].corr()

# Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Between Key Pollutants')
plt.show()


###  Step 6: Python Code to Calculate AQI from PM2.5

In [ ]:
def calculate_aqi_pm25(pm):
    breakpoints = [
        (0, 30, 0, 50),
        (30, 60, 51, 100),
        (60, 90, 101, 200),
        (90, 120, 201, 300),
        (120, 250, 301, 400),
        (250, 500, 401, 500)
    ]
    for (cl, ch, il, ih) in breakpoints:
        if cl <= pm <= ch:
            return ((ih - il) / (ch - cl)) * (pm - cl) + il
    return None
df['AQI_PM2_5'] = df['pm2_5'].apply(calculate_aqi_pm25)


In [ ]:
df[['pm2_5', 'AQI_PM2_5']].head(10)


###  Python Code to Calculate AQI from PM10

In [ ]:
def calculate_aqi_pm10(pm):
    breakpoints = [
        (0, 50, 0, 50),
        (50, 100, 51, 100),
        (100, 250, 101, 200),
        (250, 350, 201, 300),
        (350, 430, 301, 400),
        (430, 500, 401, 500)
    ]
    for (cl, ch, il, ih) in breakpoints:
        if cl <= pm <= ch:
            return ((ih - il) / (ch - cl)) * (pm - cl) + il
    return None  # If value is outside the range
df['AQI_PM10'] = df['pm10'].apply(calculate_aqi_pm10)


In [ ]:
df[['pm10', 'AQI_PM10']].head(10)


### Step 7: Plot Estimated AQI Over Time

In [ ]:
df['AQI_Estimated'] = df[['AQI_PM2_5', 'AQI_PM10']].max(axis=1)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 5))
sns.lineplot(x='date', y='AQI_Estimated', data=df)
plt.title("Estimated AQI Trend (PM2.5 & PM10)")
plt.ylabel("Estimated AQI")
plt.xlabel("Date")
plt.show()


### Step 8: Conclusions & Insights

#### Top Contributors:
- PM2.5 and PM10 show the highest pollutant concentrations and are strongly correlated with each other.

- These two fine particulate matters are the primary contributors to poor air quality in Delhi.

- Other pollutants like NO₂ and SO₂ show moderate correlations but are secondary in impact compared to particulates.

#### Seasonal Impact:
- Winter exhibits the worst air quality, with spikes in both PM2.5 and PM10 levels.

- This is largely due to temperature inversion, low wind speeds, and increased use of fossil fuels (e.g., heating, biomass burning).

- Post-monsoon periods also show elevated particulate levels, likely due to crop residue burning in nearby states.

#### Policy Recommendations:
- Enhance vehicular emissions control through stricter BS-VI compliance and traffic management in the pre-winter months.

- Ban or restrict biomass and garbage burning, especially in late autumn and early winter.

- Encourage mass adoption of public transport and EV incentives.

- Install air purifiers in public spaces and schools during winter months.

- Promote real-time air quality alerts to inform the public and reduce outdoor activity during high pollution days.